In [ ]:
# from pathlib import Path
# import os
# import sys

# PROJECT_ROOT = Path.cwd().parent

# sys.path.insert(0, str(PROJECT_ROOT))

# os.environ.setdefault(
#     "DJANGO_SETTINGS_MODULE",
#     "FinantialEv_v1.settings"
# )

# os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

# import django
# django.setup()

# print(PROJECT_ROOT)


e:\Users\jmartin\Desktop\PricingProjects\EvaluadorFinanciero_v1\FinantialEv_v1


Modelo predictivo de precios:

In [7]:
from pathlib import Path
import joblib

BASE_DIR = Path.cwd().parent          # Si el notebook está en /notebooks
MODEL_PATH = BASE_DIR / "Backend" / "ml_models" / "rf_vlr_mbps_v2.pkl"

package = joblib.load(MODEL_PATH)

model = package["model"]
features = package["features"]

print(features)

['LOG_CAP', 'Longitud', 'Latitud']


Análisis del modelo predictivo de precios

In [6]:
import numpy as np
import pandas as pd
import geopandas as gpd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path

BASE_DIR = Path.cwd().parent

df = pd.read_csv(BASE_DIR / "municipalities.csv", delimiter=';',decimal=",")
gdf = gpd.read_file("colombia-municipios.geojson")

gdf["MPIO_CDPMP"] = gdf["MPIO_CDPMP"].astype(int)
df["dane code"] = df["dane code"].astype(int)

In [3]:
def predecir_colombia(capacidad):
    df_pred = df.copy()

    df_pred["LOG_CAP"] = np.log10(capacidad)

    df_pred["Longitud"] = df_pred["longitude"]
    df_pred["Latitud"] = df_pred["latitude"]
    
    df_pred["LOG_VLR"] = model.predict(
        df_pred[features]
    )
    df_pred["VLR_MBPS"] = np.power(
        10,
        df_pred["LOG_VLR"]
    )

    mapa = gdf.merge(
        df_pred,
        left_on="MPIO_CDPMP",
        right_on="dane code",
        how="left"
    )

    return mapa

In [ ]:
# import plotly.io as pio
# pio.renderers.default = "browser"
# print(pio.renderers.default)

def dibujar_mapa(capacidad):

    mapa = predecir_colombia(capacidad)

    fig = px.choropleth(
        mapa,
        geojson=mapa.geometry.__geo_interface__,
        locations=mapa.index,
        color="VLR_MBPS",
        hover_name="MPIO_CNMBR",
        hover_data={
            "VLR_MBPS":":,.0f",
            "region":True,
            "node":True,
            "MPIO_CDPMP":True
        },
        color_continuous_scale="Viridis"
    )

    fig.update_geos(
        fitbounds="locations",
        visible=False
    )

    fig.update_layout(
        title=f"Valor predicho por Mbps ({capacidad:,} Mbps)",
        margin=dict(
            l=0,
            r=0,
            t=50,
            b=0
        )
    )

    fig.show()

browser


In [14]:
dibujar_mapa(1500)

In [43]:
slider = widgets.IntSlider(
    value=1000,
    min=50,
    max=10000,
    step=50,
    description="Mbps",
    continuous_update=False
)

In [44]:
widgets.interactive_output(
    dibujar_mapa,
    {
        "capacidad": slider
    }
)

display(slider)

widgets.interactive_output(
    dibujar_mapa,
    {
        "capacidad": slider
    }
)

IntSlider(value=1000, continuous_update=False, description='Mbps', max=10000, min=50, step=50)

Output()